# Week 4: Dimensional Modelling - Structural Walkthrough

You'll climb this like a staircase, one step at a time:

1. Start with the analytical problem.
2. Learn the modelling language.
3. Learn the core dimensional patterns.
4. Place those patterns in enterprise architecture.
5. Build a minimal star schema.
6. Extend it with a second fact table and a conformed dimension.
7. End with business consumption: semantic layer and charts.

You won't get lost in raw data engineering here. Your goal is to understand **why dimensional modelling exists**, and how each modelling choice you make solves a specific reporting problem.

### Prerequisites
- Week 3 introduced the bronze -> silver medallion. Here we run that same pattern ourselves (Lab 0 + Lab 1) and then build the **gold** dimensional model - the Gold step Week 3 said Week 4 would cover. Internet is needed only on the very first run (to download bronze).
- Comfortable with basic SQL (`JOIN`, `GROUP BY`) and pandas DataFrames.

### Learning objectives
By the end of this notebook you can:
- Explain the difference between facts and dimensions, and why the split matters.
- Declare the **grain** of a fact table and say why it must be stated explicitly.
- Classify a measure as additive, semi-additive, or non-additive and aggregate it correctly.
- Build a star schema with surrogate keys and contrast it with a snowflake.
- Use a conformed dimension to compare two business processes on one timeline.
- Describe SCD Type 1 vs Type 2 and where the semantic layer fits.
        


_Build: v5 (medallion) - generated 2026-06-03 21:27_


## Part 1 - Start With a Real Problem

A policy analyst at the municipality asks one simple-sounding question:

> **"Which municipalities have the highest household income, and how does that relate to labour-market vacancies over time?"**

The raw CBS data is not built to answer this. It is a pile of wide, operational tables: region names, region types, year labels, sector codes, population, income, and vacancy counts all mixed together across multiple files.

When you try to answer the question directly on that raw data, the same problems appear every single time:

| # | Problem in the raw data | What it costs you |
|---|--------------------------|-------------------|
| P1 | Too many joins for a simple question | Slow, fragile, 100+ line queries |
| P2 | Metrics and descriptive labels live in the same wide table | No predictable query shape |
| P3 | Mixed granularity (neighborhood rows next to municipality rows) | Wrong totals, double counting |
| P4 | Unstable / reused source identifiers | Joins silently break across years |
| P5 | History gets overwritten when regions merge or rename | Cannot answer "as of 2017" |
| P6 | Two business processes (income vs vacancies) cannot be compared | No shared timeline |
| P7 | Every analyst re-invents the same definitions | Different people, different answers |

**The rest of this notebook is organized around these problems.** Each dimensional modelling concept below is introduced together with the exact problem(s) from this list that it removes. By the end, every problem P1-P7 is solved.

Dimensional modelling exists to turn this messy source data into something stable, queryable, and explainable - and to do that organizing work **once**, in the warehouse, instead of once per analyst.
        


## Part 2 - Core Modelling Language

**Goal.** Split one messy wide table into **measures** and **context** so every analytical query has an obvious shape.

The wide raw table mixes the numbers we want to aggregate with the labels we want to filter and group by. Dimensional modelling forces a more useful question: *what are the numbers, and what are the labels?*

### The four-step design recipe (use this every time)
Kimball's design process is always the same four steps, **in order**:

1. **Pick the business process** - what real-world activity are we measuring? (CBS measures each region, each year.)
2. **Declare the grain** - what does *exactly one* fact row represent? (One row per region per year.)
3. **Choose the dimensions** - what describes that row (who / what / where / when)? (region, year.)
4. **Choose the facts** - what numbers do we record at that grain? (population, income.)

The grain (step 2) is the hinge: dimensions and facts only make sense once you have said what one row means. Everything below is one of these steps done well.

### Facts vs Dimensions - the core split
Every wide table is split into two roles:
- **Dimensions** = the context (the labels)
- **Facts** = the measurements (the numbers)

**Problems it solves:**
- **P1 (too many joins):** queries follow one predictable star shape instead of chaining many wide tables.
- **P2 (metrics tangled with context):** numbers and labels are physically separated, so every query has an obvious structure.
- **P7 (everyone rebuilds logic):** the split is defined once in the model, not re-derived per analyst.

### Dimensions = Context
Dimensions describe the **who / what / where / when** around the measurement.

Examples: region, year, sector. A dimension row answers descriptive questions such as *which municipality is this?*, *which year?*, *which sector code?*

**Problems it solves:**
- **P2 (mixed metrics and labels):** all descriptive attributes live in one clearly-labelled place.
- **P7 (inconsistent definitions):** one agreed label for each municipality / year / sector, reused everywhere.

### Facts = Measurements
Facts hold the measurable event or observation. A fact row mostly contains foreign keys pointing to dimensions plus the measures we aggregate.

Examples: population, average income, vacancy count.

**Problems it solves:**
- **P2 (metrics tangled with context):** the measurement is stored once and reused across many slicing perspectives.
- **P1 (heavy queries):** business users analyze one number from many angles without rewriting the logic each time.

### Measures
Measures are the numeric values we aggregate (sum, average, trend over time).

Not every measure can be aggregated the same way. This is one of the most common beginner mistakes, so learn the three types and which aggregation each one allows:
- **Additive** - can be summed across *every* dimension. Example: `population`, `vacancy count`. `SUM` is always safe.
- **Semi-additive** - can be summed across some dimensions but not time. Example: an account balance or a population *snapshot* (summing the same region across years is meaningless).
- **Non-additive** - cannot be summed at all. Example: `avg_income_eur` (a ratio/average). Summing or averaging an average across regions is statistically wrong - a correct municipality figure must be **population-weighted**.

**Problems it solves:**
- **P2 (hard-to-aggregate numbers):** classifying each measure tells you exactly which aggregation (`SUM`, weighted `AVG`, or none) is valid, so totals and trends are trustworthy.

### Grain
The grain defines exactly what one row in a fact table means - for example *one row per region per year* or *one row per sector per quarter*.

Grain is a *decision you declare first*, not something you discover later. In our lab the CBS source contains both neighborhood (`Buurt`) and municipality (`Gemeente`) rows, so the grain is **one row per region (at any level) per year**. Because a municipality already contains its neighborhoods, you must filter to a single `region_type` before summing - otherwise you double count. That filter is the guardrail the grain forces on you.

**Problems it solves:**
- **P3 (mixed granularity):** a declared grain makes the mixed levels explicit and tells you that querying always picks one level.
- **P3 (double counting):** with one clear meaning per row plus a level filter, totals are unambiguous.

### Surrogate Keys
Surrogate keys are warehouse-generated integer keys that replace unstable natural source identifiers.

**Problems it solves:**
- **P4 (unstable / reused source keys):** joins use a stable internal key, so they do not break when source codes change.
- **P5 (history gets overwritten):** a region can have multiple surrogate keys over time, which makes historical versions possible.
        


## Part 3 - Core Dimensional Patterns

### Star Schema
One fact table in the center, directly joined to denormalized dimensions.

**Problems it solves:**
- **P1 (too many joins):** every question is one fact joined to a few dimensions - no long join chains.
- **P2 (no predictable shape):** all analytics share the same readable star pattern.
- **P7 (BI is hard):** the simple shape is easy for BI tools and business users to consume.

### Snowflake Schema
Dimensions are split into smaller normalized tables (e.g. region -> municipality).

**Problems it solves:**
- **dimension value duplication:** repeated descriptive values are stored once in a normalized sub-table.
- **hierarchy maintenance:** formal parent/child hierarchies are modelled explicitly.

**Tradeoff:** star is usually easier for analytics; snowflake adds joins and complexity. We show the contrast so you can feel the cost directly.

### Slowly Changing Dimensions (SCD)
Descriptive attributes change over time (regions merge, rename). How you handle that change is the SCD type:
- **Type 1 - overwrite:** replace the old value. Simple, but history is lost - you can no longer answer "what was it in 2017?".
- **Type 2 - versioned rows:** keep the old row and add a new one with `valid_from` / `valid_to` / `is_current` flags. History is preserved.

**Problems it solves:**
- **P5 (history overwritten):** Type 2 preserves old attribute values as historical rows (Type 1 does not).
- **"as of date" questions:** with Type 2 you can reconstruct what a region looked like in any past year.

### Conformed Dimensions
One shared dimension (here `dim_year`) reused by multiple fact tables.

**Problems it solves:**
- **P6 (processes cannot be compared):** income and vacancies share the same time dimension, so they sit on one timeline.
- **P7 (inconsistent definitions across marts):** a single agreed definition is reused, enabling trustworthy drill-across.
        


## Part 4 - Enterprise Context

These approaches answer a bigger problem: **how do you organize the whole warehouse, not just one star?** Each makes a different tradeoff.

### Kimball
- dimensional marts first; fastest path to reporting value.
- **Problem it addresses:** slow time-to-value - business needs answers before a full enterprise model exists.

### Inmon
- normalized enterprise warehouse first; slower but more centralized.
- **Problem it addresses:** fragmented, inconsistent marts - it enforces one central, integrated definition of the business.

### Data Vault 2.0
- hubs, links, satellites.
- **Problem it addresses:** weak auditability and lineage - it tracks where every value came from and when, for reproducibility and compliance.

### Semantic Layer
The semantic layer sits above the star schema and gives business-friendly names, trusted measures, and reusable definitions.

**Problems it solves:**
- **P7 (different analysts, different answers):** one trusted definition of each measure, reused by everyone.
- **users rewrite logic:** common calculations are defined once and exposed as named measures.

```text
Raw data -> Silver -> Gold dimensional model -> Semantic layer -> Dashboards
```

This is where the warehouse stops being just tables and becomes a business product.
        


## Part 5 - Lab Overview

We build the **full medallion** (bronze -> silver -> gold) inside the notebook, then model on top of it.

### Lab 0 - Bronze
Download the raw CBS responses, untouched, into `week4/data/bronze` (the landing zone).

### Lab 1 - Silver
Clean, type and project the bronze JSON into parquet in `week4/data/silver`.

### Lab 2 - Minimal Star
Build a simple neighborhood-income star schema from local CBS parquet files.

### Lab 3 - Shared-Dimension Extension
Add a vacancies fact table and reuse `dim_year` as a conformed dimension.

### Lab 4 - Consumption
Create semantic views and simple charts that show why the model exists.

The target shape for Lab 2 and Lab 3 is a two-fact star sharing one time dimension:

```mermaid
erDiagram
    dim_region ||--o{ fact_neighborhood_stats : describes
    dim_year   ||--o{ fact_neighborhood_stats : describes
    dim_year   ||--o{ fact_vacancies : describes
    dim_sector ||--o{ fact_vacancies : describes
    fact_neighborhood_stats {
        int region_key FK
        int year_key FK
        int population
        double avg_income_eur
    }
    fact_vacancies {
        int year_key FK
        int sector_key FK
        string quarter
        double vacancies_seasonadj
    }
```

`dim_year` sits between both facts - that is the conformed dimension (P6) you will build.

### The bus matrix
List **business processes** down the side and **conformed dimensions** across the top; an `X` marks which dimension each process shares. That grid is the **Kimball bus matrix** - the map of the whole warehouse on one page:

| Business process (fact) | dim_region | dim_year | dim_sector |
|---|---|---|---|
| Neighbourhood stats | X | X | |
| Job vacancies | | X | X |

The shared `dim_year` column is exactly what makes the drill-across in Lab 3 legal. Add a new fact later (e.g. job postings) and you simply add a **row**.
        


In [ ]:
from pathlib import Path

import json

import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import requests

# --- Medallion folders (same vocabulary as Week 3: bronze -> silver -> gold) ---
DATA = Path("week4/data")
BRONZE, SILVER, GOLD = DATA / "bronze", DATA / "silver", DATA / "gold"
for _d in (BRONZE, SILVER, GOLD):
    _d.mkdir(parents=True, exist_ok=True)

con = duckdb.connect()
print("Medallion folders ready under", DATA.as_posix())
        


## Lab 0 - Bronze: Land the Raw Data

Following the Week 3 medallion pattern, **bronze is the raw landing zone**: an exact, unmodified copy of what the CBS OData API returns, saved as JSON in `week4/data/bronze`. We never transform bronze - it is our replay and audit insurance. This step is idempotent: if a file is already on disk, we skip the download.
        


In [ ]:
# CBS column rename maps. The codes drift between years - that is schema evolution.
_COMMON = {
    "WijkenEnBuurten": "region_code", "SoortRegio_2": "region_type",
    "Gemeentenaam_1": "municipality", "AantalInwoners_5": "population",
    "Bevolkingsdichtheid_33": "pop_density",
}
_VERSIONS = {
    2017: ("83765NED", {"GemiddeldInkomenPerInwoner_66": "avg_income_x1000", "MateVanStedelijkheid_104": "urbanization_class"}),
    2019: ("84583NED", {"GemiddeldInkomenPerInwoner_72": "avg_income_x1000", "MateVanStedelijkheid_115": "urbanization_class"}),
    2021: ("85039NED", {"GemiddeldInkomenPerInwoner_72": "avg_income_x1000", "MateVanStedelijkheid_116": "urbanization_class"}),
}


def fetch_cbs(code, page=5000):
    """Page through a CBS OData TypedDataSet and return the raw row dicts."""
    rows, skip = [], 0
    while True:
        batch = requests.get(
            f"https://opendata.cbs.nl/ODataFeed/odata/{code}/TypedDataSet",
            params={"$top": page, "$skip": skip, "$format": "json"}, timeout=60,
        ).json()["value"]
        if not batch:
            break
        rows += batch
        if len(batch) < page:
            break
        skip += page
    return rows


for year, (code, _) in _VERSIONS.items():
    dst = BRONZE / f"neighborhoods_{year}.json"
    if dst.exists():
        print(f"  bronze skip  {dst.name}")
        continue
    print(f"  bronze fetch {code} -> {dst.name}")
    dst.write_text(json.dumps(fetch_cbs(code), ensure_ascii=False), encoding="utf-8")

if not (BRONZE / "vacancies_sbi.json").exists():
    (BRONZE / "vacancies_sbi.json").write_text(json.dumps(fetch_cbs("80474NED"), ensure_ascii=False), encoding="utf-8")
if not (BRONZE / "vacancy_sectors.json").exists():
    sectors = requests.get(
        "https://opendata.cbs.nl/ODataFeed/odata/80474NED/SBI2008PartBedrijvenOverheid?$format=json", timeout=30
    ).json()["value"]
    (BRONZE / "vacancy_sectors.json").write_text(json.dumps(sectors, ensure_ascii=False), encoding="utf-8")

print("Bronze ready:", sorted(p.name for p in BRONZE.glob("*.json")))


## Lab 1 - Silver: Clean & Conform

**Silver** applies schema-on-read to bronze: rename the cryptic CBS codes to stable names, cast types, derive `avg_income_eur`, and project to the columns we model. The output is typed parquet in `week4/data/silver` - the single source the Gold layer reads. Idempotent: skip if the parquet already exists.
        


In [ ]:
for year, (_, renames) in _VERSIONS.items():
    dst = SILVER / f"neighborhoods_{year}.parquet"
    if dst.exists():
        print(f"  silver skip  {dst.name}")
        continue
    raw = json.loads((BRONZE / f"neighborhoods_{year}.json").read_text(encoding="utf-8"))
    df = pd.DataFrame(raw).rename(columns={**_COMMON, **renames})
    for c in ("region_code", "region_type", "municipality"):
        df[c] = df[c].astype(str).str.strip()
    for c in ("population", "pop_density", "avg_income_x1000", "urbanization_class"):
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    df["year"] = year
    df["avg_income_eur"] = (df["avg_income_x1000"] * 1000).round(0)
    keep = ["year", "region_code", "region_type", "municipality", "population",
            "pop_density", "urbanization_class", "avg_income_eur"]
    df[keep].to_parquet(dst, index=False)
    print(f"  silver build {dst.name}")

vac_dst = SILVER / "vacancies_sbi.parquet"
if not vac_dst.exists():
    raw = json.loads((BRONZE / "vacancies_sbi.json").read_text(encoding="utf-8"))
    v = pd.DataFrame(raw).rename(columns={
        "SBI2008PartBedrijvenOverheid": "sbi_code", "Perioden": "period_code",
        "VacaturesSeizoengecorrigeerd_1": "vacancies_seasonadj",
    })
    v["sbi_code"] = v["sbi_code"].astype(str).str.strip()
    v["vacancies_seasonadj"] = pd.to_numeric(v["vacancies_seasonadj"], errors="coerce")
    v["year"] = pd.to_numeric(v["period_code"].str.slice(0, 4), errors="coerce").astype("Int64")
    v["quarter"] = pd.to_numeric(v["period_code"].str.slice(6, 8), errors="coerce").astype("Int64")
    v[["year", "quarter", "period_code", "sbi_code", "vacancies_seasonadj"]].to_parquet(vac_dst, index=False)

sec_dst = SILVER / "vacancy_sectors.parquet"
if not sec_dst.exists():
    raw = json.loads((BRONZE / "vacancy_sectors.json").read_text(encoding="utf-8"))
    s = pd.DataFrame(raw).rename(columns={"Key": "sbi_code", "Title": "sector_title"})
    s["sbi_code"] = s["sbi_code"].astype(str).str.strip()
    s["sector_title"] = s["sector_title"].astype(str).str.strip()
    s[["sbi_code", "sector_title"]].to_parquet(sec_dst, index=False)

# Paths the Gold labs read, plus the working silver frame
year_files = {y: SILVER / f"neighborhoods_{y}.parquet" for y in (2017, 2019, 2021)}
vacancy_fact_path = SILVER / "vacancies_sbi.parquet"
vacancy_sector_path = SILVER / "vacancy_sectors.parquet"
print("Silver ready:", sorted(p.name for p in SILVER.glob("*.parquet")))


## Lab 2 - Step 1: Load the Silver Layer for Modelling

The Gold layer reads **silver**. We pull the cleaned parquet into one working frame, keeping the columns the model needs and the two region levels we use (`Buurt`, `Gemeente`).
        


In [ ]:
frames = []

for year, path in year_files.items():
    query = f"""
        SELECT
            {year} AS year,
            region_code,
            region_type,
            municipality,
            population,
            avg_income_eur
        FROM read_parquet('{path.as_posix()}')
        WHERE region_code IS NOT NULL
          AND region_type IN ('Buurt', 'Gemeente')
          AND municipality IS NOT NULL
          AND population IS NOT NULL
          AND avg_income_eur IS NOT NULL
    """
    frames.append(con.execute(query).df())

silver_neighborhood = pd.concat(frames, ignore_index=True)
silver_neighborhood.head()
        


In [ ]:
# Visual: the mixed-grain trap - one CBS file holds several region levels at once
grain_counts = con.execute(
    f"SELECT region_type, COUNT(*) AS n FROM read_parquet('{year_files[2021].as_posix()}') WHERE region_type IS NOT NULL GROUP BY region_type ORDER BY n DESC"
).df()

plt.figure(figsize=(8, 4))
plt.bar(grain_counts["region_type"], grain_counts["n"], color="#156082")
plt.yscale("log")
for i, n in enumerate(grain_counts["n"]):
    plt.text(i, n, f"{n:,}", ha="center", va="bottom", fontsize=9, color="#334155")
plt.title("One CBS file, several region grains (2021, log scale)")
plt.ylabel("rows (log scale)")
plt.tight_layout()
plt.show()


## Lab 2 - Step 2: Build the Dimension Tables

We start with the smallest useful dimensional model:
- `dim_region`
- `dim_year`

These dimensions answer the descriptive side of the business question.
        


In [ ]:
dim_region = (
    silver_neighborhood[["region_code", "region_type", "municipality"]]
    .drop_duplicates()
    .sort_values(["region_type", "municipality", "region_code"])
    .reset_index(drop=True)
)
dim_region.insert(0, "region_key", range(1, len(dim_region) + 1))

dim_year = pd.DataFrame(
    {
        "year_key": [1, 2, 3],
        "year": [2017, 2019, 2021],
    }
)

display(dim_region.head())
display(dim_year)
        


## Lab 2 - Step 3: Build the Fact Table

The fact table contains mostly keys and measures.

**Declared grain:** `one row per region (any level) per year`.

Because the source mixes `Buurt` and `Gemeente` rows, queries must filter to one `region_type` before summing additive measures - we verify this guardrail in the next step.
        


In [ ]:
fact_neighborhood_stats = (
    silver_neighborhood
    .merge(dim_region, on=["region_code", "region_type", "municipality"], how="left")
    .merge(dim_year, on="year", how="left")
    [["region_key", "year_key", "population", "avg_income_eur"]]
    .sort_values(["year_key", "region_key"])
    .reset_index(drop=True)
)

# Referential integrity check: every fact row must map to a real dimension key.
# A null key here means a join failed - exactly the P4 problem surrogate keys guard against.
assert fact_neighborhood_stats["region_key"].notna().all(), "Unmatched region_key found"
assert fact_neighborhood_stats["year_key"].notna().all(), "Unmatched year_key found"
print("Referential integrity OK: all fact rows have valid dimension keys.")

fact_neighborhood_stats.head()
        


## Lab 2 - Step 4: Ask the First Proper Question

This is the test of the star schema. If the design is good, the query becomes small and readable.

Notice two things below: you filter to `region_type = 'Gemeente'` (the grain guardrail, so you never mix levels), and because `avg_income_eur` is **non-additive**, the statistically correct municipality figure is a **population-weighted** average, not a plain `AVG`.
        


In [ ]:
con.register("dim_region", dim_region)
con.register("dim_year", dim_year)
con.register("fact_neighborhood_stats", fact_neighborhood_stats)

top_income_2021 = con.execute(
    """
    SELECT
        y.year,
        r.municipality,
        SUM(f.avg_income_eur * f.population) / NULLIF(SUM(f.population), 0) AS avg_income_eur,
        SUM(f.population) AS total_population
    FROM fact_neighborhood_stats f
    JOIN dim_region r ON r.region_key = f.region_key
    JOIN dim_year y ON y.year_key = f.year_key
    WHERE y.year = 2021
      AND r.region_type = 'Gemeente'
    GROUP BY y.year, r.municipality
    ORDER BY avg_income_eur DESC
    LIMIT 10
    """
).df()

top_income_2021
        


## Lab 3 - Step 5: Add a Second Fact Table

Now conformed dimensions become real instead of theoretical.

We will build:
- `dim_sector`
- `fact_vacancies`

and reuse `dim_year`. Note that `fact_vacancies` keeps `quarter` and `period_code` as plain columns on the fact row. These are **degenerate dimensions** - dimension-like attributes with no separate table (there is nothing else to describe a quarter). Promoting `quarter` to its own dimension is the first "Your Turn" task.
        


In [ ]:
dim_sector = con.execute(
    f"""
    SELECT DISTINCT
        sbi_code,
        sector_title
    FROM read_parquet('{vacancy_sector_path.as_posix()}')
    WHERE sbi_code IS NOT NULL
      AND sector_title IS NOT NULL
    ORDER BY sbi_code
    """
).df()

dim_sector.insert(0, "sector_key", range(1, len(dim_sector) + 1))
dim_sector.head()
        


In [ ]:
vacancy_base = con.execute(
    f"""
    SELECT
        year,
        quarter,
        period_code,
        sbi_code,
        vacancies_seasonadj
    FROM read_parquet('{vacancy_fact_path.as_posix()}')
    WHERE year IN (2017, 2019, 2021)
      AND sbi_code IS NOT NULL
      AND vacancies_seasonadj IS NOT NULL
    """
).df()

fact_vacancies = (
    vacancy_base
    .merge(dim_year, on="year", how="left")
    .merge(dim_sector[["sector_key", "sbi_code"]], on="sbi_code", how="left")
    [["year_key", "quarter", "period_code", "sector_key", "vacancies_seasonadj"]]
    .sort_values(["year_key", "sector_key", "quarter"])
    .reset_index(drop=True)
)

fact_vacancies.head()
        


## Lab 3 - Step 6: Prove the Conformed Dimension

Both facts now share `dim_year`. That lets us compare different business processes on the same timeline.
        


In [ ]:
con.register("dim_sector", dim_sector)
con.register("fact_vacancies", fact_vacancies)

income_by_year = con.execute(
    """
    SELECT
        y.year,
        SUM(f.avg_income_eur * f.population) / NULLIF(SUM(f.population), 0) AS avg_income_eur
    FROM fact_neighborhood_stats f
    JOIN dim_year y ON y.year_key = f.year_key
    JOIN dim_region r ON r.region_key = f.region_key
    WHERE r.region_type = 'Gemeente'
    GROUP BY y.year
    ORDER BY y.year
    """
).df()

vacancies_by_year = con.execute(
    """
    SELECT
        y.year,
        AVG(f.vacancies_seasonadj) AS avg_vacancies
    FROM fact_vacancies f
    JOIN dim_year y ON y.year_key = f.year_key
    GROUP BY y.year
    ORDER BY y.year
    """
).df()

yearly_comparison = income_by_year.merge(vacancies_by_year, on="year", how="inner")
yearly_comparison
        


## Lab 3 - Step 7: Short Snowflake Contrast

To make the star vs snowflake tradeoff concrete, we split region into a tiny hierarchy and compare the query shape.

Snowflake isn't "bad" - you build it here only so you can feel the extra join complexity for yourself.
        


In [ ]:
dim_municipality = (
    dim_region[["municipality"]]
    .drop_duplicates()
    .sort_values("municipality")
    .reset_index(drop=True)
)
dim_municipality.insert(0, "municipality_key", range(1, len(dim_municipality) + 1))

dim_region_snowflake = dim_region.merge(dim_municipality, on="municipality", how="left")
dim_region_snowflake = dim_region_snowflake[["region_key", "region_code", "region_type", "municipality_key"]]

display(dim_municipality.head())
display(dim_region_snowflake.head())
        


In [ ]:
star_query = con.execute(
    """
    SELECT
        r.municipality,
        AVG(f.avg_income_eur) AS avg_income_eur
    FROM fact_neighborhood_stats f
    JOIN dim_region r ON r.region_key = f.region_key
    JOIN dim_year y ON y.year_key = f.year_key
    WHERE y.year = 2021 AND r.region_type = 'Gemeente'
    GROUP BY r.municipality
    ORDER BY avg_income_eur DESC
    LIMIT 5
    """
).df()

con.register("dim_municipality", dim_municipality)
con.register("dim_region_snowflake", dim_region_snowflake)

snowflake_query = con.execute(
    """
    SELECT
        m.municipality,
        AVG(f.avg_income_eur) AS avg_income_eur
    FROM fact_neighborhood_stats f
    JOIN dim_region_snowflake r ON r.region_key = f.region_key
    JOIN dim_municipality m ON m.municipality_key = r.municipality_key
    JOIN dim_year y ON y.year_key = f.year_key
    WHERE y.year = 2021 AND r.region_type = 'Gemeente'
    GROUP BY m.municipality
    ORDER BY avg_income_eur DESC
    LIMIT 5
    """
).df()

print("Star query result:")
display(star_query)
print("Snowflake query result:")
display(snowflake_query)
        


## Lab 4 - Step 8: Slowly Changing Dimensions on a Real Merger

Descriptive attributes change over time. On **2019-01-01** the municipalities **Bedum**, **Winsum** and **De Marne** merged into a new municipality, **Het Hogeland** (`GM1966`); their old codes vanish from the 2021 data.

We *detect* the change straight from the CBS files, then model it. **Type 1 (overwrite)** keeps only the 2021 snapshot, so Bedum is gone forever. **Type 2 (versioned)** keeps the old rows with `valid_from` / `valid_to` / `is_current`, so you can still ask "what was true in 2018?".
        


In [ ]:
# Detect the real 2019 merger straight from the CBS files
g17 = con.execute(
    f"SELECT DISTINCT region_code, municipality FROM read_parquet('{year_files[2017].as_posix()}') WHERE region_type = 'Gemeente'"
).df()
g21 = con.execute(
    f"SELECT DISTINCT region_code, municipality FROM read_parquet('{year_files[2021].as_posix()}') WHERE region_type = 'Gemeente'"
).df()
removed = g17[~g17["region_code"].isin(g21["region_code"])]
added = g21[~g21["region_code"].isin(g17["region_code"])]

print("Merged away by 2021 (the three originals):")
display(removed[removed["municipality"].isin(["Bedum", "Winsum", "De Marne"])])
print("New municipality in 2021:")
display(added[added["municipality"] == "Het Hogeland"])

# SCD Type 1 (overwrite): only the 2021 snapshot survives - Bedum is gone forever
print("\nSCD Type 1 - searching for Bedum (overwritten, so absent):")
display(g21[g21["municipality"] == "Bedum"])

# SCD Type 2 (versioned): close the three originals, open the merger result - real codes from the data
closed = removed[removed["municipality"].isin(["Bedum", "Winsum", "De Marne"])].copy()
closed["valid_from"], closed["valid_to"], closed["is_current"] = "2017-01-01", "2018-12-31", False
opened = added[added["municipality"] == "Het Hogeland"].copy()
opened["valid_from"], opened["valid_to"], opened["is_current"] = "2019-01-01", None, True
scd_type2 = pd.concat([closed, opened], ignore_index=True)[
    ["region_code", "municipality", "valid_from", "valid_to", "is_current"]
]
print("\nSCD Type 2 - the merger preserved with validity windows:")
scd_type2
        


## Lab 4 - Step 9: Build the Semantic Layer

The semantic layer gives business-friendly access to the warehouse. It standardizes labels and derived measures so users do not keep reinventing definitions.
        


In [ ]:
con.execute(
    """
    CREATE OR REPLACE VIEW vw_region_income AS
    SELECT
        y.year AS report_year,
        r.region_type AS region_level,
        r.municipality,
        f.population AS residents,
        f.avg_income_eur AS average_income_eur,
        ROUND(f.avg_income_eur / NULLIF(f.population, 0), 6) AS income_per_resident_ratio
    FROM fact_neighborhood_stats f
    JOIN dim_region r ON r.region_key = f.region_key
    JOIN dim_year y ON y.year_key = f.year_key
    """
)

con.execute(
    """
    CREATE OR REPLACE VIEW vw_yearly_labor_income AS
    SELECT
        i.year,
        i.avg_income_eur,
        v.avg_vacancies
    FROM (
        SELECT y.year, SUM(f.avg_income_eur * f.population) / NULLIF(SUM(f.population), 0) AS avg_income_eur
        FROM fact_neighborhood_stats f
        JOIN dim_year y ON y.year_key = f.year_key
        JOIN dim_region r ON r.region_key = f.region_key
        WHERE r.region_type = 'Gemeente'
        GROUP BY y.year
    ) i
    JOIN (
        SELECT y.year, AVG(f.vacancies_seasonadj) AS avg_vacancies
        FROM fact_vacancies f
        JOIN dim_year y ON y.year_key = f.year_key
        GROUP BY y.year
    ) v ON v.year = i.year
    """
)

con.execute("SELECT * FROM vw_yearly_labor_income ORDER BY year").df()
        


## Lab 4 - Step 10: Charts

The warehouse exists to support consumption. These plots make the payoff visible.
        


In [ ]:
plot_top = top_income_2021.sort_values("avg_income_eur")

plt.figure(figsize=(10, 6))
plt.barh(plot_top["municipality"], plot_top["avg_income_eur"], color="#156082")
plt.title("Top Municipalities by Average Income in 2021")
plt.xlabel("Average Income (EUR)")
plt.ylabel("Municipality")
plt.tight_layout()
plt.show()
        


In [ ]:
plot_yearly = con.execute("SELECT * FROM vw_yearly_labor_income ORDER BY year").df()

fig, ax1 = plt.subplots(figsize=(10, 6))
ax1.plot(plot_yearly["year"], plot_yearly["avg_income_eur"], marker="o", color="#156082", label="Average Income")
ax1.set_xlabel("Year")
ax1.set_ylabel("Average Income (EUR)")

ax2 = ax1.twinx()
ax2.plot(plot_yearly["year"], plot_yearly["avg_vacancies"], marker="s", color="#94A3B8", label="Average Vacancies")
ax2.set_ylabel("Average Vacancies")

plt.title("Income and Vacancies Over Time")
fig.tight_layout()
plt.show()
        


In [ ]:
# Visual: busiest sectors in 2021 (the second fact, via dim_sector)
top_sectors = con.execute(
    """
    SELECT s.sector_title, AVG(f.vacancies_seasonadj) AS avg_vac
    FROM fact_vacancies f
    JOIN dim_sector s ON s.sector_key = f.sector_key
    JOIN dim_year y ON y.year_key = f.year_key
    WHERE y.year = 2021
    GROUP BY s.sector_title
    ORDER BY avg_vac DESC
    LIMIT 8
    """
).df().sort_values("avg_vac")

plt.figure(figsize=(10, 5))
plt.barh(top_sectors["sector_title"], top_sectors["avg_vac"], color="#156082")
plt.title("Busiest sectors by average vacancies, 2021")
plt.xlabel("Avg vacancies (x1000)")
plt.tight_layout()
plt.show()


In [ ]:
# Visual: avg_income is NON-ADDITIVE - naive AVG of buurten vs population-weighted
import numpy as np

compare = con.execute(
    """
    SELECT r.municipality,
           AVG(f.avg_income_eur) AS naive_avg,
           SUM(f.avg_income_eur * f.population) / NULLIF(SUM(f.population), 0) AS weighted_avg
    FROM fact_neighborhood_stats f
    JOIN dim_region r ON r.region_key = f.region_key
    JOIN dim_year y ON y.year_key = f.year_key
    WHERE y.year = 2021 AND r.region_type = 'Buurt'
    GROUP BY r.municipality
    ORDER BY weighted_avg DESC
    LIMIT 8
    """
).df()

ypos = np.arange(len(compare))
plt.figure(figsize=(10, 5))
plt.barh(ypos - 0.2, compare["naive_avg"], height=0.4, color="#94A3B8", label="naive AVG (wrong)")
plt.barh(ypos + 0.2, compare["weighted_avg"], height=0.4, color="#156082", label="population-weighted (correct)")
plt.yticks(ypos, compare["municipality"])
plt.gca().invert_yaxis()
plt.title("Non-additive measure: naive vs population-weighted income (2021)")
plt.xlabel("Average income (EUR)")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Bonus: one sector's full vacancy history (the transactional grain, all years)
traj = con.execute(
    f"""
    SELECT v.year + (CAST(v.quarter AS DOUBLE) - 1) / 4 AS t, v.vacancies_seasonadj AS vac
    FROM read_parquet('{vacancy_fact_path.as_posix()}') v
    JOIN read_parquet('{vacancy_sector_path.as_posix()}') s ON s.sbi_code = v.sbi_code
    WHERE s.sector_title = 'G Handel' AND v.vacancies_seasonadj IS NOT NULL
    ORDER BY t
    """
).df()

plt.figure(figsize=(10, 4))
plt.plot(traj["t"], traj["vac"], color="#156082", linewidth=1.6)
plt.title("'G Handel' vacancies, full history (transactional grain)")
plt.xlabel("Year")
plt.ylabel("Vacancies (x1000)")
plt.tight_layout()
plt.show()


## Lab 4 - Step 11: Persist the Gold Layer

The gold tables are the business-ready dimensional model. We write them to `week4/data/gold` as parquet so downstream tools (BI, the semantic layer, Week 5) can read a clean star without re-running the pipeline.
        


In [ ]:
gold_tables = {
    "dim_region": dim_region,
    "dim_year": dim_year,
    "dim_sector": dim_sector,
    "fact_neighborhood_stats": fact_neighborhood_stats,
    "fact_vacancies": fact_vacancies,
}
for name, frame in gold_tables.items():
    frame.to_parquet(GOLD / f"{name}.parquet", index=False)

print("Gold ready:", sorted(p.name for p in GOLD.glob("*.parquet")))


## Your Turn

1. Extend `fact_vacancies` with a quarter dimension.
2. Build a sector-level semantic view.
3. Create one chart comparing sectors instead of years.
4. Explain in your own words why `dim_year` is a conformed dimension.

If you can complete those steps, you understand the structural purpose of dimensional modelling rather than only the syntax.
        
